# SciEntsBank: Dataset Audit

**Question:** What does one record represent, and what data-quality issues must be understood before modelling?

This independent portfolio study examines public English science short answers. It starts from education experience and focuses on reproducible data inspection. One row represents an answer record, not a unique learner. This release contains data analysis only.

Source: [Dzikovska et al. (2013)](https://aclanthology.org/S13-2045/) and the [author-hosted release](https://github.com/myrosia/semeval-2013-task7). Data and data-derived outputs: CC BY-SA 3.0.

In [1]:
from pathlib import Path
import subprocess
import sys

# Works from this checkout, a local Jupyter session, or a fresh Colab session.
roots = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in roots if (p / "data/raw/scientsbank_core.zip").exists()), None)
if ROOT is None:
    checkout = Path("/content/scientsbank_portfolio")
    if not checkout.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/Junghwan0516/early-learning-support.git",
                        str(checkout)], check=True)
    ROOT = checkout / "projects/science-answer-benchmark"
sys.path.insert(0, str(ROOT / "scripts"))
import json
import pandas as pd
from IPython.display import display, Image
from build_analysis import build
print(f"Project: {ROOT.name}")

Project: science-answer-benchmark


## 1. Rebuild the tables and descriptive figures
All descriptive figures use the official training split. Other official splits are exported and checked for inventory and exact normalized input overlap only. No prediction model is fitted.

In [2]:
audit = build()
display(pd.Series(audit, name="value").to_frame())

{
  "training_answers": 4969,
  "training_questions": 135,
  "training_domains": 12,
  "minority_answers": 23,
  "minority_questions": 19,
  "minority_domains": 9,
  "unique_normalized_training_inputs": 4826,
  "duplicate_groups": 82,
  "duplicate_excess_rows": 143,
  "conflicting_input_groups": 6,
  "conflicting_rows": 15,
  "train_ua_overlap_rows": 25,
  "blank_fields": 0,
  "median_answer_words": 11.0,
  "source_sha256": "57c14e5c260d8cfd774f3fb954985d94bcfa3407029bfe585179650b907dbf9b"
}


,value
training_answers,4969
training_questions,135
training_domains,12
minority_answers,23
minority_questions,19
minority_domains,9
unique_normalized_training_inputs,4826
duplicate_groups,82
duplicate_excess_rows,143
conflicting_input_groups,6


## 2. Inspect one question
Read the question and reference before interpreting its student-answer labels.

In [3]:
train = pd.read_csv(ROOT / "data/processed/train.csv")
sample = train[train.question_id.eq("EM_45b")]
print("QUESTION:", sample.question_text.iloc[0])
print("REFERENCE:", sample.reference_answer.iloc[0])
display(sample[["answer_id", "student_answer", "label"]].head(10))

QUESTION: You used several methods to separate and identify the substances in mock rocks. How did you separate the salt from the water?
REFERENCE: The water was evaporated, leaving the salt.


,answer_id,student_answer,label
0,EM.45b.110.1,By letting it sit in a dish for a day.,irrelevant
1,EM.45b.113.1,Let the water evaporate and the salt is left b...,correct
2,EM.45b.114.1,The water evaporated and left salt crystals.,correct
3,EM.45b.261.1,I saw a pinkish grayish color that was blockin...,irrelevant
4,EM.45b.383.1,You have to slowly tip the vial for only the w...,irrelevant
5,EM.45b.384.1,"By pouring the water and salt into the thing, ...",correct
6,EM.45b.385.1,By slowly pouring it in a tray.,irrelevant
7,EM.45b.395.1,The water evaporated so there was only salt left.,correct
8,EM.45b.402.1,We put the water with the salt in it and put i...,correct
9,EM.45b.416.1,I separated the salt from the water because th...,irrelevant


## 3. Official split inventory
UA reuses the 135 training questions; UQ holds out questions; UD holds out three domain codes. The counts below are not performance measurements.

In [4]:
display(pd.read_csv(ROOT / "reports/tables/split_manifest.csv"))

,official_split,answers,questions,domains,unique_normalized_inputs
0,train,4969,135,12,4826
1,test-unseen-answers,540,135,12,539
2,test-unseen-questions,733,15,11,703
3,test-unseen-domains,4562,46,3,4229


## 4. Check missing fields and label counts

In [5]:
display(pd.read_csv(ROOT / "reports/tables/missing_values.csv"))
display(pd.read_csv(ROOT / "reports/tables/label_distribution.csv"))

,column,null_count,blank_count
0,question_id,0,0
1,answer_id,0,0
2,domain,0,0
3,question_text,0,0
4,reference_answer,0,0
5,student_answer,0,0
6,label,0,0


,label,answer_count,percentage
0,correct,2008,40.410545
1,partially_correct_incomplete,1324,26.645200
2,irrelevant,1115,22.439123
3,contradictory,499,10.042262
4,non_domain,23,0.462870


## 5. Examine the minority class
Only 23 non-domain records occur across 19 questions and 9 domains. These small supports constrain future evaluation.

In [6]:
display(pd.read_csv(ROOT / "reports/tables/minority_domain_distribution.csv"))
display(pd.read_csv(ROOT / "reports/tables/minority_question_distribution.csv"))
display(train.loc[train.label.eq("non_domain"), ["domain", "question_id", "student_answer"]])

,domain,total_answers,total_questions,non_domain_count,non_domain_percentage
0,EM,430,12,7,1.627907
1,FN,323,9,1,0.309598
2,II,213,6,0,0.000000
3,LF,393,11,1,0.254453
4,LP,70,2,0,0.000000
5,ME,828,23,5,0.603865
6,MS,252,7,2,0.793651
7,MX,697,20,3,0.430416
8,PS,545,11,2,0.366972
9,SE,539,15,1,0.185529


,domain,question_id,non_domain_count
0,EM,EM_26,2
1,EM,EM_35,2
2,EM,EM_45b,1
3,EM,EM_45c,1
4,EM,EM_48b,1
5,FN,FN_17c,1
6,LF,LF_6b,1
7,ME,ME_17b,1
8,ME,ME_17d,1
9,ME,ME_27b,1


,domain,question_id,student_answer
30,EM,EM_45b,I do not know.
47,EM,EM_45c,The teacher told us.
252,EM,EM_48b,I chose this answer because I remember what ty...
317,EM,EM_26,I do not know.
318,EM,EM_26,I do not know.
382,EM,EM_35,I do not know.
387,EM,EM_35,I do not know.
666,FN,FN_17c,I was not here when we did it.
1330,LF,LF_6b,Because I saw it.
1454,ME,ME_27b,You would find the answer.


## 6. Inspect duplicate inputs and conflicting labels
Input identity uses Unicode NFKC, case folding, and whitespace collapsing on the question, reference, and student answer. It preserves punctuation and word order. Original text and original labels remain unchanged. A repeated input can be a genuine separate submission; repetition is not automatically an error.

In [7]:
duplicates = pd.read_csv(ROOT / "reports/tables/duplicate_groups.csv")
conflicts = pd.read_csv(ROOT / "reports/tables/conflicting_labels.csv")
print("Duplicate groups:", len(duplicates))
print("Conflicting input groups:", conflicts.normalized_input_hash.nunique())
display(conflicts[["question_id", "student_answer", "label", "answer_id"]])

Duplicate groups: 82
Conflicting input groups: 6


,question_id,student_answer,label,answer_id
0,EM_21a,Rub them together.,correct,EM.21a.572.1
1,EM_21a,Rub them together.,partially_correct_incomplete,EM.21a.185.1
2,EM_21a,Rub them together.,partially_correct_incomplete,EM.21a.575.1
3,EM_21a,Rub them together.,partially_correct_incomplete,EM.21a.632.1
4,EM_21a,Georgia could rub them together.,correct,EM.21a.529.1
5,EM_21a,Georgia could rub them together.,partially_correct_incomplete,EM.21a.435.1
6,SE_4a,When the temperature drops.,contradictory,SE.4a.408.1
7,SE_4a,When the temperature drops.,irrelevant,SE.4a.384.1
8,ME_17c,It will work.,correct,ME.17c.146.1
9,ME_17c,It will work.,irrelevant,ME.17c.224.1


## 7. Record split overlap
There are 25 UA records whose normalized complete input also occurs in training. Keep the official benchmark intact. A future model report should present an additional non-overlapping slice without choosing training changes from test outcomes.

In [8]:
display(pd.read_csv(ROOT / "reports/tables/train_test_overlap.csv"))

,official_split,answers,matching_training_input_rows,shared_question_ids
0,test-unseen-answers,540,25,135
1,test-unseen-questions,733,0,0
2,test-unseen-domains,4562,0,0


## Interpretation and next questions

- No null or blank values were found in the seven checked training fields. This is not a claim that all labels are correct.
- The 4,969 records reduce to 4,826 normalized input groups. There are 82 repeated groups, 143 excess rows, and 6 label-conflicting groups comprising 15 records.
- Rare-class examples are spread across questions, but future splits still need explicit class-support checks.
- No labels have been changed, and no records have been deduplicated.
- Future modelling needs question-level development splits, transparent duplicate handling, and separate UA/UQ/UD reporting.

**Status:** data audit complete; model development deferred.